# Visualización Ejecutiva y Análisis de Impacto (Estratégico)

## Objetivos
- Visualizar arquitecturas para audiencias ejecutivas (impacto en negocio, no detalles técnicos).
- Aplicar **Value Stream Mapping** para identificar cuellos de botella en flujos de datos.
- Diagramas de decisiones arquitectónicas (ADRs) para comunicar trade-offs.
- Capability maps: desde estrategia de negocio hasta arquitectura de datos.
- Cuantificar impacto: ROI, time-to-insight, reducción de costos.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Estrategia Multi-Cloud: Migración, Costos y Gobierno Avanzado](07_estrategia_multi_cloud_migracion_costos.ipynb) | Siguiente: [Transformación Cultural hacia Data-Driven: Estrategia Ejecutiva](09_transformacion_cultural_data_driven_estrategia.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. Objetivos
2. 1. Visualización para Ejecutivos: Principios
3. 2. Value Stream Mapping (VSM) para Datos
4. 3. Código: Simulador de VSM
5. 4. Diagramas de Decisiones Arquitectónicas (ADRs)
6. 5. Capability Maps: De Estrategia a Arquitectura
7. 6. Cuantificación de Impacto
8. 7. Código: Calculadora de Impacto

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Visualización Ejecutiva y Análisis de Impacto (Estratégico)"] --> C["Notebook siguiente"]
```


## 1. Visualización para Ejecutivos: Principios
- **Simplicidad:** una sola idea por diagrama.
- **Impacto de negocio:** traducir arquitectura a KPIs (costo, tiempo, riesgo).
- **Sin jerga técnica:** reemplazar "Kafka" por "Sistema de Eventos Tiempo Real".
- **Colores para prioridad:** verde (bajo riesgo), amarillo (medio), rojo (alto).

### Ejemplo: Migración Cloud
```mermaid
graph LR
    A[Data Center On-Prem<br/>Costo: $200K/año<br/>Riesgo: Alto] -->|Migración| B[Cloud Híbrido<br/>Costo: $150K/año<br/>Riesgo: Medio]
    B -->|Optimización| C[Cloud Full<br/>Costo: $100K/año<br/>Riesgo: Bajo]
    
    style A fill:#f99,stroke:#333
    style B fill:#ff9,stroke:#333
    style C fill:#9f9,stroke:#333
```

**Mensaje:** reducción de 50% en costos, menor riesgo.

## 2. Value Stream Mapping (VSM) para Datos
Metodología Lean para identificar desperdicios en flujos.

### Componentes
- **Lead Time:** tiempo total desde fuente hasta insight.
- **Process Time:** tiempo efectivo de procesamiento (sin esperas).
- **Wait Time:** esperas, colas, aprobaciones.
- **Efficiency:** Process Time / Lead Time.

### Ejemplo: Pipeline de Reportes Regulatorios
| Etapa              | Process Time | Wait Time | Lead Time |
|-------------------|-------------|----------|----------|
| Extracción fuentes | 2h          | 12h      | 14h      |
| Validación calidad | 1h          | 24h      | 25h      |
| Transformación ETL | 3h          | 0h       | 3h       |
| Aprobación legal   | 0.5h        | 48h      | 48.5h    |
| Publicación        | 0.5h        | 0h       | 0.5h     |
| **Total**          | **7h**      | **84h**  | **91h**  |

**Efficiency:** 7h / 91h = **7.7%** → 92% del tiempo es espera.

### Oportunidades
- Paralelizar extracción y validación (reduce 12h).
- Automatizar aprobación legal con reglas (reduce 47h).
- **Nuevo Lead Time potencial:** 32h (65% reducción).

## 3. Código: Simulador de VSM

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class Stage:
    name: str
    process_time_h: float
    wait_time_h: float
    
    @property
    def lead_time_h(self) -> float:
        return self.process_time_h + self.wait_time_h

def calculate_vsm_metrics(stages: List[Stage]) -> dict:
    total_process = sum(s.process_time_h for s in stages)
    total_wait = sum(s.wait_time_h for s in stages)
    total_lead = total_process + total_wait
    efficiency = (total_process / total_lead * 100) if total_lead > 0 else 0
    
    return {
        "total_process_h": total_process,
        "total_wait_h": total_wait,
        "total_lead_h": total_lead,
        "efficiency_pct": round(efficiency, 2)
    }

# Ejemplo actual
current_stages = [
    Stage("Extracción", 2, 12),
    Stage("Validación", 1, 24),
    Stage("ETL", 3, 0),
    Stage("Aprobación", 0.5, 48),
    Stage("Publicación", 0.5, 0)
]

# Escenario optimizado
optimized_stages = [
    Stage("Extracción+Validación", 3, 0),  # Paralelo
    Stage("ETL", 3, 0),
    Stage("Aprobación Auto", 0.5, 1),  # Reglas automáticas
    Stage("Publicación", 0.5, 0)
]

current = calculate_vsm_metrics(current_stages)
optimized = calculate_vsm_metrics(optimized_stages)

print("=== Current State ===")
print(f"Lead Time: {current['total_lead_h']}h")
print(f"Efficiency: {current['efficiency_pct']}%")
print()
print("=== Optimized State ===")
print(f"Lead Time: {optimized['total_lead_h']}h")
print(f"Efficiency: {optimized['efficiency_pct']}%")
print(f"Improvement: {round((1 - optimized['total_lead_h']/current['total_lead_h'])*100, 1)}% reduction")

## 4. Diagramas de Decisiones Arquitectónicas (ADRs)
**ADR** = Architecture Decision Record.

### Visualización de Trade-offs
```mermaid
graph TD
    Decision[Elegir tecnología de streaming]
    Decision --> Kafka[Apache Kafka]
    Decision --> Kinesis[AWS Kinesis]
    Decision --> Pulsar[Apache Pulsar]
    
    Kafka --> KafkaP[+ Open source<br/>+ Comunidad amplia<br/>- Ops complejas]
    Kinesis --> KinesisP[+ Managed AWS<br/>+ Integración nativa<br/>- Vendor lock-in]
    Pulsar --> PulsarP[+ Multi-tenancy<br/>+ Geo-replicación<br/>- Menos maduro]
    
    style Kafka fill:#9f9
    style Kinesis fill:#ff9
    style Pulsar fill:#f99
```

**Decisión documentada:** Kafka (verde) por balance entre capacidad y costo operativo.

## 5. Capability Maps: De Estrategia a Arquitectura
Mapas visuales que conectan objetivos de negocio → capacidades → componentes técnicos.

### Ejemplo: Banco Digital
| Objetivo Negocio         | Capacidad Requerida          | Componente Arquitectura       |
|--------------------------|------------------------------|------------------------------|
| Reducir fraude 30%       | Detección tiempo real        | Streaming analytics (Kafka+Flink) |
| Personalización ofertas  | Segmentación avanzada        | Data Lakehouse (Delta+Spark)  |
| Cumplimiento GDPR        | Trazabilidad consentimiento  | Data Catalog + Lineage       |
| Reducir costos TI 20%    | Optimización cloud           | FinOps + Reserved Instances  |

**Visualización:**
```mermaid
graph TD
    O1[Reducir Fraude 30%] --> C1[Detección Tiempo Real]
    C1 --> T1[Kafka + Flink]
    
    O2[Personalización] --> C2[Segmentación]
    C2 --> T2[Delta Lakehouse]
    
    O3[GDPR] --> C3[Trazabilidad]
    C3 --> T3[Data Catalog]
    
    O4[Reducir Costos 20%] --> C4[FinOps]
    C4 --> T4[Reserved Instances]
    
    style O1 fill:#bbf
    style O2 fill:#bbf
    style O3 fill:#bbf
    style O4 fill:#bbf
```

## 6. Cuantificación de Impacto
Traducir arquitectura a métricas de negocio.

### Tabla de Impacto: Migración a Lakehouse
| Métrica                  | Antes (DW tradicional) | Después (Lakehouse) | Mejora  |
|--------------------------|-----------------------|---------------------|--------|
| Costo mensual storage    | $50K                  | $20K                | -60%   |
| Time-to-insight          | 3 días                | 4 horas             | -95%   |
| Cobertura datos (%)      | 40%                   | 85%                 | +112%  |
| Usuarios con acceso self-service | 20        | 150                 | +650%  |
| Incidentes calidad/mes   | 15                    | 3                   | -80%   |

**ROI estimado:** $360K ahorros anuales, payback en 18 meses.

## 7. Código: Calculadora de Impacto

In [ ]:
from dataclasses import dataclass

@dataclass
class ImpactMetrics:
    cost_before: float
    cost_after: float
    time_to_insight_before_h: float
    time_to_insight_after_h: float
    coverage_before_pct: float
    coverage_after_pct: float
    
    def cost_reduction_pct(self) -> float:
        return round((1 - self.cost_after / self.cost_before) * 100, 1)
    
    def time_improvement_pct(self) -> float:
        return round((1 - self.time_to_insight_after_h / self.time_to_insight_before_h) * 100, 1)
    
    def coverage_increase_pct(self) -> float:
        return round((self.coverage_after_pct - self.coverage_before_pct) / self.coverage_before_pct * 100, 1)
    
    def annual_savings(self) -> float:
        return (self.cost_before - self.cost_after) * 12

# Ejemplo: Migración Lakehouse
lakehouse_impact = ImpactMetrics(
    cost_before=50_000,
    cost_after=20_000,
    time_to_insight_before_h=72,  # 3 días
    time_to_insight_after_h=4,
    coverage_before_pct=40,
    coverage_after_pct=85
)

print("=== Impacto Migración Lakehouse ===")
print(f"Reducción de costo: {lakehouse_impact.cost_reduction_pct()}%")
print(f"Mejora time-to-insight: {lakehouse_impact.time_improvement_pct()}%")
print(f"Aumento cobertura: {lakehouse_impact.coverage_increase_pct()}%")
print(f"Ahorro anual: ${lakehouse_impact.annual_savings():,.0f}")
print()
print("Presentación ejecutiva:")
print(f"- Reducimos costos de storage en {lakehouse_impact.cost_reduction_pct()}%")
print(f"- Insights 18x más rápidos ({lakehouse_impact.time_to_insight_before_h/lakehouse_impact.time_to_insight_after_h:.0f}x)")
print(f"- Duplicamos la cobertura de datos para decisiones")

## 8. Ejercicio Integrativo: Caso Banco
**Contexto:** Banco regional con 1M clientes, DW on-prem saturado, objetivo: lanzar banca móvil personalizada.

**Tareas:**
1. Crea un capability map: desde "Aumentar uso app móvil 50%" hasta componentes arquitectura.
2. Diseña VSM actual vs. futuro para pipeline de recomendaciones (desde transacción → modelo ML → app).
3. Genera ADR visual para decisión: Lakehouse propio vs. Databricks managed.
4. Calcula impacto: asume $80K/mes on-prem, $50K/mes cloud, time-to-insight 5 días → 2 horas.
5. Prepara 1 slide ejecutivo con diagrama de impacto (costo, tiempo, coverage).

## 9. Resumen
- **Visualización ejecutiva:** impacto > detalles técnicos.
- **Value Stream Mapping:** identifica desperdicios (wait time) en flujos de datos.
- **ADR diagrams:** comunican trade-offs de decisiones arquitectónicas.
- **Capability maps:** conectan estrategia de negocio con tecnología.
- **Cuantificación:** ROI, time-to-insight, cobertura traducen arquitectura a lenguaje de negocio.

**Aplicación:** estas herramientas respaldan casos de inversión (business case) para proyectos de arquitectura.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- C4 Model: https://c4model.com/
- Referencias del curso: ../04-recursos/referencias.md
